# Paired DCP and identity baseline comparison (Kaggle, inference only)

Compare the observed hazy image (identity), a frozen Dark Channel Prior (DCP) implementation, and three fixed Stage 4 checkpoints on paired O-HAZE (primary; all 45 pairs) and, when present, the pooled 55-pair Dense-Haze release (secondary; not an official challenge-only subset). All methods use the same 512×512 RGB inputs, radiometric preprocessing, ground truth, and evaluator-compatible PSNR, SSIM, MS-SSIM, MAE, and mean CIEDE2000 conventions. CIEDE2000 is reported as unavailable if `scikit-image` is absent.

DCP is fixed before evaluation: patch 15×15 rectangular minimum filter; atmospheric-light candidates are the top 0.1% dark-channel pixels, selecting the candidate with greatest RGB sum; `omega=0.95`; transmission floor `t0=0.10`; no guided-filter refinement. This is a transparent classical reference, not a tuned comparator. It follows the Dark Channel Prior formulation of [He, Sun, and Tang (CVPR 2009)](https://ieeexplore.ieee.org/document/5206515).

The comparison is descriptive evidence for these paired releases and checkpoints. It does not isolate component effects, establish state-of-the-art performance, or support deployment claims. Dense-Haze is pooled-release context. No training or evaluation-set parameter tuning occurs. The image-cluster bootstrap resamples images and retains all three checkpoint-seed outcomes for each sampled image; intervals are conditional on these fixed datasets and checkpoints.

Edit Kaggle paths in the next cell if your attached datasets use different locations. Run cells in order. The final cell zips all generated outputs.

## Planning runtime on Kaggle 2×T4

Budget roughly **5–15 minutes for O-HAZE only** and **10–25 minutes if the optional pooled Dense-Haze set is also available**. These are planning estimates, not measured timings. The notebook makes 135 model-image predictions for O-HAZE (45 images × 3 checkpoints), or 300 across both sets (100 images × 3 checkpoints), plus fixed-baseline scoring, metric calculation, checkpoint loading, bootstrap summaries, and plots. Inference is sequential and does not use a multi-GPU distribution strategy, so it will likely run on one visible T4; having two T4s should not be assumed to halve the runtime. Actual time depends on Kaggle hardware allocation, TensorFlow startup/model loading, and GPU contention. The scoring cell prints its own elapsed time; that excludes summary/bootstrap/plot generation and ZIP creation. The final ZIP includes every regular file currently under the output directory, including earlier files if you rerun without clearing that folder.


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import time

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

# Kaggle input paths; configurable for a different attachment name.
PACKAGE = Path("/kaggle/input/datasets/randomguy28/reviewer-test-packge/reviewer_test_package")
OHAZE_ROOT = Path("/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY")
DENSE_ROOT = Path("/kaggle/input/datasets/lucky02655/den-haze-test")
OUTPUT_ROOT = Path("/kaggle/working/paired_dcp_identity_baseline")
IMAGE_SIZE = 512
SEEDS = (17, 43, 73)
BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_SEED = 20261014
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
METRICS = ("psnr", "ssim", "ms_ssim", "mae", "deltaE2000")
BASELINES = ("identity", "dcp")

assert PACKAGE.is_dir(), f"Missing reviewer package: {PACKAGE}"
assert (OHAZE_ROOT / "hazy").is_dir(), f"Missing O-HAZE hazy directory: {OHAZE_ROOT / 'hazy'}"
assert (OHAZE_ROOT / "GT").is_dir(), f"Missing O-HAZE GT directory: {OHAZE_ROOT / 'GT'}"
SCRIPT_ROOT = PACKAGE / "code"
sys.path.insert(0, str(SCRIPT_ROOT))
from eca_ldnet.data import get_gt_path, load_image_pair
from eca_ldnet.evaluate import predict_full
from eca_ldnet.model import load_pretrained

CHECKPOINTS = {
    seed: PACKAGE / f"models/full/seed{seed}/stage4_best.keras"
    for seed in SEEDS
}
missing_checkpoints = [str(path) for path in CHECKPOINTS.values() if not path.is_file()]
assert not missing_checkpoints, f"Missing fixed Stage 4 checkpoint(s): {missing_checkpoints}"

DCP_PARAMETERS = {
    "patch_size": 15,
    "omega": 0.95,
    "transmission_floor": 0.10,
    "atmospheric_top_fraction": 0.001,
    "atmospheric_candidate_rule": "greatest RGB sum among top dark-channel pixels",
    "refinement": "none",
    "minimum_filter": "OpenCV rectangular erosion, 15x15, replicate border",
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("Visible GPUs:", [device.name for device in tf.config.list_physical_devices("GPU")])
print("Output directory:", OUTPUT_ROOT)


In [ ]:
def collect_pairs(root, dataset_name, expected_count=None):
    hazy_dir, gt_dir = root / "hazy", root / "GT"
    if not hazy_dir.is_dir() or not gt_dir.is_dir():
        return [], {"status": "skipped", "reason": "hazy or GT directory missing", "root": str(root)}
    files = sorted(path for path in hazy_dir.iterdir() if path.suffix.lower() in IMAGE_EXTS)
    pairs, unmatched = [], []
    for hazy_path in files:
        gt_path = get_gt_path(hazy_path.name, str(gt_dir))
        if gt_path is None:
            unmatched.append(hazy_path.name)
        else:
            pairs.append({"image": hazy_path.name, "hazy_path": str(hazy_path), "gt_path": str(gt_path)})
    info = {"status": "available", "root": str(root), "hazy_files": len(files),
            "matched_pairs": len(pairs), "unmatched_hazy": unmatched}
    if expected_count is not None and len(pairs) != expected_count:
        info["status"] = "skipped"
        info["reason"] = f"expected {expected_count} matched pairs, found {len(pairs)}"
        return [], info
    if unmatched:
        info["status"] = "skipped"
        info["reason"] = "one or more hazy files lack a matched ground truth"
        return [], info
    return pairs, info


def load_dataset_pairs(pairs, dataset_name, required=False):
    loaded, unreadable = [], []
    for pair in pairs:
        hazy, clean = load_image_pair(
            pair["hazy_path"], pair["gt_path"], size=IMAGE_SIZE, preprocessing="radiometric"
        )
        if hazy is None or clean is None or hazy.shape != clean.shape:
            unreadable.append(pair["image"])
            continue
        loaded.append({"image": pair["image"], "hazy": hazy, "clean": clean,
                       "hazy_path": pair["hazy_path"], "gt_path": pair["gt_path"]})
    if unreadable and required:
        raise RuntimeError(f"Unreadable/mismatched required {dataset_name} pair(s): {unreadable}")
    return loaded, unreadable

ohaze_pairs, ohaze_info = collect_pairs(OHAZE_ROOT, "O-HAZE", expected_count=45)
assert ohaze_info["status"] == "available", f"O-HAZE must include all 45 pairs: {ohaze_info}"

dense_pairs, dense_info = collect_pairs(DENSE_ROOT, "Dense-Haze pooled release", expected_count=55)
if dense_info["status"] == "skipped":
    print("Dense-Haze skipped:", dense_info["reason"])

DATASET_PAIRS = {"O-HAZE": ohaze_pairs}
if dense_info["status"] == "available":
    DATASET_PAIRS["Dense-Haze pooled release (55)" ] = dense_pairs

DATASETS = {}
UNREADABLE = {}
for dataset_name, pairs in DATASET_PAIRS.items():
    loaded, unreadable = load_dataset_pairs(pairs, dataset_name, required=(dataset_name == "O-HAZE"))
    if unreadable and dataset_name != "O-HAZE":
        dense_info["status"] = "skipped"
        dense_info["reason"] = f"unreadable/mismatched pairs: {unreadable}"
        print("Dense-Haze skipped:", dense_info["reason"])
        continue
    if dataset_name == "O-HAZE":
        assert len(loaded) == 45, f"Expected 45 readable O-HAZE pairs, got {len(loaded)}"
    DATASETS[dataset_name] = loaded
    UNREADABLE[dataset_name] = unreadable
    print(f"{dataset_name}: {len(loaded)} readable pairs")


In [ ]:
def dark_channel(image, patch_size=15):
    channel_min = np.min(np.asarray(image, dtype=np.float32), axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (patch_size, patch_size))
    return cv2.erode(channel_min, kernel, borderType=cv2.BORDER_REPLICATE)


def dark_channel_prior(image, parameters=DCP_PARAMETERS):
    """Frozen single-image DCP estimate; input/output RGB float32 in [0, 1]."""
    image = np.clip(np.asarray(image, dtype=np.float32), 0.0, 1.0)
    height, width, _ = image.shape
    dark = dark_channel(image, parameters["patch_size"])
    flat_dark = dark.reshape(-1)
    top_n = max(1, int(np.ceil(parameters["atmospheric_top_fraction"] * flat_dark.size)))
    candidate_indices = np.argpartition(flat_dark, -top_n)[-top_n:]
    brightness = image.reshape(-1, 3)[candidate_indices].sum(axis=1)
    atmospheric = image.reshape(-1, 3)[candidate_indices[int(np.argmax(brightness))]]
    atmospheric = np.maximum(atmospheric, 1.0 / 255.0)
    normalized = image / atmospheric.reshape(1, 1, 3)
    normalized_dark = dark_channel(normalized, parameters["patch_size"])
    transmission = np.clip(1.0 - parameters["omega"] * normalized_dark, 0.0, 1.0)
    transmission_safe = np.maximum(transmission, parameters["transmission_floor"])
    restored = (image - atmospheric.reshape(1, 1, 3)) / transmission_safe[..., None] + atmospheric.reshape(1, 1, 3)
    return np.clip(restored, 0.0, 1.0).astype(np.float32)


try:
    from skimage.color import deltaE_ciede2000, rgb2lab
    HAS_DELTA_E = True
except ImportError:
    HAS_DELTA_E = False
    print("scikit-image missing: CIEDE2000 will be NaN; other metrics remain available.")


def score_prediction(prediction, clean):
    pred = np.clip(np.asarray(prediction, dtype=np.float32), 0.0, 1.0)
    gt = np.clip(np.asarray(clean, dtype=np.float32), 0.0, 1.0)
    diff = pred.astype(np.float64) - gt.astype(np.float64)
    scores = {
        "psnr": float(np.asarray(tf.image.psnr(pred[None], gt[None], 1.0).numpy()).reshape(-1)[0]),
        "ssim": float(np.asarray(tf.image.ssim(pred[None], gt[None], 1.0).numpy()).reshape(-1)[0]),
        "mae": float(np.mean(np.abs(diff))),
        "mse": float(np.mean(np.square(diff))),
        "rmse": float(np.sqrt(np.mean(np.square(diff)))),
    }
    try:
        scores["ms_ssim"] = float(np.asarray(tf.image.ssim_multiscale(pred[None], gt[None], 1.0).numpy()).reshape(-1)[0])
    except (tf.errors.InvalidArgumentError, ValueError):
        scores["ms_ssim"] = float("nan")
    if HAS_DELTA_E:
        scores["deltaE2000"] = float(np.mean(deltaE_ciede2000(rgb2lab(pred), rgb2lab(gt))))
    else:
        scores["deltaE2000"] = float("nan")
    return scores

print("DCP parameters frozen:", json.dumps(DCP_PARAMETERS, sort_keys=True))
print("CIEDE2000 available:", HAS_DELTA_E)


## Score all fixed methods

The identity and DCP predictions are computed once per image. Each checkpoint is loaded once and evaluated on all available image pairs. No model or DCP parameter is selected from these evaluation scores. Checkpoint outputs are clipped to `[0,1]`, matching the paired evaluator convention.


In [ ]:
score_rows = []
checkpoint_hashes = {}
run_started = time.perf_counter()

for dataset_name, records in DATASETS.items():
    print(f"Scoring {dataset_name}: n={len(records)}")
    # Baselines are fixed functions of each hazy input, independent of model seed.
    for index, record in enumerate(records, start=1):
        hazy, clean, image_name = record["hazy"], record["clean"], record["image"]
        for method, prediction in (("identity", hazy), ("dcp", dark_channel_prior(hazy))):
            score_rows.append({"dataset": dataset_name, "image": image_name,
                               "method": method, "seed": "baseline", **score_prediction(prediction, clean)})
        if index % 10 == 0 or index == len(records):
            print(f"  baseline: {index}/{len(records)} images")

    # Load each fixed checkpoint once per dataset, then score every image.
    for seed, checkpoint in CHECKPOINTS.items():
        digest = hashlib.sha256()
        with checkpoint.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        checkpoint_hashes[seed] = {"path": str(checkpoint), "sha256": digest.hexdigest()}
        print(f"  loading fixed Stage 4 seed {seed}")
        model = load_pretrained(str(checkpoint), compile=False)
        model_shape = model.input_shape
        if model_shape[1] is not None and model_shape[1] != IMAGE_SIZE:
            raise ValueError(f"Seed {seed} checkpoint expects height {model_shape[1]}, not {IMAGE_SIZE}")
        if model_shape[2] is not None and model_shape[2] != IMAGE_SIZE:
            raise ValueError(f"Seed {seed} checkpoint expects width {model_shape[2]}, not {IMAGE_SIZE}")
        for index, record in enumerate(records, start=1):
            prediction = predict_full(model, record["hazy"])
            score_rows.append({"dataset": dataset_name, "image": record["image"],
                               "method": f"model_seed_{seed}", "seed": seed,
                               **score_prediction(prediction, record["clean"])})
            if index % 10 == 0 or index == len(records):
                print(f"    seed {seed}: {index}/{len(records)} images")
        del model
        tf.keras.backend.clear_session()

scores = pd.DataFrame(score_rows)
assert not scores.empty
expected_rows = sum(len(records) for records in DATASETS.values()) * (2 + len(SEEDS))
assert len(scores) == expected_rows, f"Expected {expected_rows} rows; got {len(scores)}"
scores_path = OUTPUT_ROOT / "per_image_scores.csv"
scores.to_csv(scores_path, index=False)
print(f"Saved {len(scores):,} rows to {scores_path}")
print(f"Elapsed: {(time.perf_counter() - run_started) / 60:.1f} min")


In [ ]:
# Method-level descriptive summaries. Model seeds remain separately visible.
summary_rows = []
for (dataset, method), group in scores.groupby(["dataset", "method"], sort=False):
    for metric in METRICS:
        values = group[metric].to_numpy(dtype=float)
        finite = values[np.isfinite(values)]
        summary_rows.append({"dataset": dataset, "method": method, "metric": metric,
                             "n_images": int(len(finite)), "mean": float(np.mean(finite)) if len(finite) else np.nan,
                             "std": float(np.std(finite, ddof=1)) if len(finite) > 1 else np.nan})
method_summary = pd.DataFrame(summary_rows)
method_summary.to_csv(OUTPUT_ROOT / "dataset_method_summary.csv", index=False)

# Paired per-image/model-seed differences against both baselines.
paired_rows = []
model_rows = scores[scores["method"].str.startswith("model_seed_")]
for _, model_row in model_rows.iterrows():
    for baseline in BASELINES:
        baseline_rows = scores[(scores["dataset"] == model_row["dataset"])
                               & (scores["image"] == model_row["image"])
                               & (scores["method"] == baseline)]
        if len(baseline_rows) != 1:
            raise RuntimeError(f"Expected one {baseline} row for {model_row['dataset']} / {model_row['image']}")
        base = baseline_rows.iloc[0]
        row = {"dataset": model_row["dataset"], "image": model_row["image"],
               "seed": int(model_row["seed"]), "model": model_row["method"], "baseline": baseline}
        for metric in METRICS:
            row[f"model_{metric}"] = model_row[metric]
            row[f"baseline_{metric}"] = base[metric]
            row[f"delta_{metric}"] = model_row[metric] - base[metric]
        paired_rows.append(row)
paired = pd.DataFrame(paired_rows)
paired.to_csv(OUTPUT_ROOT / "paired_model_vs_baselines_per_image.csv", index=False)

# Image-cluster bootstrap: each resampled image contributes its outcomes for all three seeds.
DIRECTION = {"psnr": "higher", "ssim": "higher", "ms_ssim": "higher",
             "mae": "lower", "deltaE2000": "lower"}
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows = []
for (dataset, baseline), comparison in paired.groupby(["dataset", "baseline"], sort=False):
    for metric in METRICS:
        delta_col = f"delta_{metric}"
        matrix = comparison.pivot(index="image", columns="seed", values=delta_col).sort_index().to_numpy(dtype=float)
        if matrix.size == 0:
            continue
        # Exclude images with no finite seed outcomes; retain every finite seed
        # outcome for each remaining image as a cluster in the bootstrap.
        matrix = matrix[np.isfinite(matrix).any(axis=1)]
        if not len(matrix):
            continue
        seed_counts = np.isfinite(matrix).sum(axis=1)
        image_means = np.nanmean(matrix, axis=1)
        n_images = len(image_means)
        draws = rng.integers(0, n_images, size=(BOOTSTRAP_SAMPLES, n_images))
        boot_means = image_means[draws].mean(axis=1)
        all_deltas = comparison[delta_col].to_numpy(dtype=float)
        all_deltas = all_deltas[np.isfinite(all_deltas)]
        if not len(all_deltas):
            continue
        better = all_deltas > 0 if DIRECTION[metric] == "higher" else all_deltas < 0
        bootstrap_rows.append({
            "dataset": dataset, "baseline": baseline, "metric": metric,
            "n_images": int(n_images),
            "n_finite_seed_outcomes": int(len(all_deltas)),
            "min_finite_seed_outcomes_per_image": int(seed_counts.min()),
            "max_finite_seed_outcomes_per_image": int(seed_counts.max()),
            "mean_delta_model_minus_baseline": float(np.mean(image_means)),
            "bootstrap_ci95_low": float(np.quantile(boot_means, 0.025)),
            "bootstrap_ci95_high": float(np.quantile(boot_means, 0.975)),
            "win_fraction_image_seed": float(np.mean(better)),
            "direction_for_win": DIRECTION[metric],
            "bootstrap_unit": "image; all seed outcomes for an image kept together",
        })
bootstrap_summary = pd.DataFrame(bootstrap_rows)
bootstrap_summary.to_csv(OUTPUT_ROOT / "paired_bootstrap_summary.csv", index=False)
print("Method means (PSNR/SSIM):")
print(method_summary[method_summary["metric"].isin(["psnr", "ssim"])].to_string(index=False))
print("Paired bootstrap summary:")
print(bootstrap_summary[bootstrap_summary["metric"].isin(["psnr", "ssim"])].to_string(index=False))


In [ ]:
# Compact plots: absolute PSNR by fixed method, and paired model-minus-baseline PSNR.
method_order = ["identity", "dcp"] + [f"model_seed_{seed}" for seed in SEEDS]
datasets_present = list(DATASETS)
fig, axes = plt.subplots(1, len(datasets_present), figsize=(6 * len(datasets_present), 4), squeeze=False)
for ax, dataset in zip(axes[0], datasets_present):
    part = method_summary[(method_summary["dataset"] == dataset) & (method_summary["metric"] == "psnr")]
    part = part.set_index("method").reindex(method_order).dropna(subset=["mean"])
    ax.bar(part.index, part["mean"], color=["#9aa0a6", "#c68f38", "#4c78a8", "#72a0c1", "#9dc2d9"][:len(part)])
    ax.set_title(dataset)
    ax.set_ylabel("Mean PSNR (dB)")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "mean_psnr_by_method.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

psnr_boot = bootstrap_summary[bootstrap_summary["metric"] == "psnr"].copy()
fig, axes = plt.subplots(1, len(datasets_present), figsize=(6 * len(datasets_present), 4), squeeze=False)
for ax, dataset in zip(axes[0], datasets_present):
    part = psnr_boot[psnr_boot["dataset"] == dataset].set_index("baseline").reindex(BASELINES).dropna(subset=["mean_delta_model_minus_baseline"])
    x = np.arange(len(part))
    mean = part["mean_delta_model_minus_baseline"].to_numpy()
    low = part["bootstrap_ci95_low"].to_numpy()
    high = part["bootstrap_ci95_high"].to_numpy()
    ax.errorbar(x, mean, yerr=np.vstack([mean - low, high - mean]), fmt="o", capsize=5, color="#355c7d")
    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    ax.set_xticks(x, part.index)
    ax.set_ylabel("Model − baseline PSNR (dB)")
    ax.set_title(dataset)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "paired_psnr_differences.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

# Run manifest records source paths, fixed parameters, software, and checkpoint hashes.
dataset_manifest = {
    "O-HAZE": {**ohaze_info, "readable_pairs": len(DATASETS["O-HAZE"]),
               "images": [row["image"] for row in DATASETS["O-HAZE"]]},
    "Dense-Haze pooled release": dense_info,
}
if "Dense-Haze pooled release (55)" in DATASETS:
    dataset_manifest["Dense-Haze pooled release"].update({
        "readable_pairs": len(DATASETS["Dense-Haze pooled release (55)" ]),
        "images": [row["image"] for row in DATASETS["Dense-Haze pooled release (55)" ]],
        "interpretation": "pooled release; not official challenge-only subset",
    })
manifest = {
    "experiment": "paired identity and frozen DCP reference comparison",
    "status": "completed when this cell has run",
    "scope": "inference-only descriptive comparison; no training or evaluation-set tuning",
    "resolution": [IMAGE_SIZE, IMAGE_SIZE],
    "color_order": "RGB",
    "input_preprocessing": "radiometric [0,1], evaluator load_image_pair convention",
    "output_clipping": "[0,1]",
    "metrics": list(METRICS),
    "metric_conventions": {
        "psnr_ssim_ms_ssim": "TensorFlow tf.image, max_val=1.0",
        "mae": "mean absolute error over all RGB values",
        "deltaE2000": "mean CIEDE2000 after skimage rgb2lab on clipped RGB [0,1]",
    },
    "dcp_parameters": DCP_PARAMETERS,
    "checkpoints": checkpoint_hashes,
    "datasets": dataset_manifest,
    "bootstrap": {"samples": BOOTSTRAP_SAMPLES, "seed": BOOTSTRAP_SEED,
                   "unit": "image cluster; all three fixed model-seed outcomes retained per sampled image",
                   "interpretation": "conditional on the fixed releases and checkpoints"},
    "limitations": [
        "comparison is descriptive and does not isolate component effects",
        "does not establish state-of-the-art performance or deployment readiness",
        "Dense-Haze is the pooled 55-pair release and is secondary context",
        "DCP is one fixed classical comparator and is not tuned on evaluation images",
    ],
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "opencv": cv2.__version__,
    "gpu_devices": [device.name for device in tf.config.list_physical_devices("GPU")],
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with (OUTPUT_ROOT / "run_manifest.json").open("w", encoding="utf-8") as stream:
    json.dump(manifest, stream, indent=2)
print("Saved outputs:", sorted(path.name for path in OUTPUT_ROOT.iterdir()))


## Package outputs

Run this last cell after the score, analysis, and manifest cells complete. The archive includes every file under the output directory.


In [ ]:
import zipfile

ZIP_PATH = Path("/kaggle/working/paired_dcp_identity_baseline.zip")
OUTPUT_ROOT = Path(OUTPUT_ROOT)
if not OUTPUT_ROOT.is_dir():
    raise FileNotFoundError(f"Output directory does not exist: {OUTPUT_ROOT}")

# Snapshot every regular file recursively under OUTPUT_ROOT before creating the archive.
output_files = sorted(path for path in OUTPUT_ROOT.rglob("*") if path.is_file())
if not output_files:
    raise RuntimeError(f"No output files found under {OUTPUT_ROOT}; run the analysis cells first.")
expected_members = {path.relative_to(OUTPUT_ROOT).as_posix() for path in output_files}
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_files:
        archive.write(path, arcname=path.relative_to(OUTPUT_ROOT).as_posix())

with zipfile.ZipFile(ZIP_PATH, mode="r") as archive:
    archived_members = set(archive.namelist())
    damaged_member = archive.testzip()
if archived_members != expected_members:
    missing = sorted(expected_members - archived_members)
    unexpected = sorted(archived_members - expected_members)
    raise RuntimeError(f"Archive membership mismatch; missing={missing}, unexpected={unexpected}")
if damaged_member is not None:
    raise RuntimeError(f"Archive integrity check failed at {damaged_member}")
print(f"Verified {len(expected_members)} output files in {ZIP_PATH}")
print(f"Archive size: {ZIP_PATH.stat().st_size / (1024 * 1024):.2f} MiB")
for member in sorted(archived_members):
    print(" -", member)
